# 01 · Setup and evaluation basics
**Objectives:** configure a reproducible environment, distinguish model output from evaluation evidence, and inspect a shared result.

Run `uv sync --locked` and `uv run jupyter lab` from `labs/boto3/`. Select the Python kernel from that environment. No AWS account is needed for offline lessons. Each notebook is independent: run its cells from top to bottom.

For live experiments, copy `.env.example` to `.env`, configure your standard AWS credentials or profile, and confirm the selected model or inference-profile IDs support Converse in your region. The agent model must also support tool use. The embedding adapter expects Titan Text Embeddings. Check these in the Bedrock console before changing `LIVE` to `True`. Existing IDs are inherited examples, not a guarantee of account access. Never put keys in notebook cells.


In [ ]:
from pathlib import Path
import json
import os
import sys

# Works from labs/boto3 or its notebooks directory, including a fresh clone.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").is_file() and (p / "data/synthetic.json").is_file())
sys.path.insert(0, str(ROOT / "src"))
from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=False)
from ai_eval_labs.core import Bedrock, DATA, Result, export_results, generate, load_cases
from ai_eval_labs.metrics import Judge, check, contains

LIVE = False  # Set True only after reviewing AWS model access, prices, and request counts.
api = Bedrock(live=LIVE)
judge = Judge(api)
synthetic = json.loads((DATA / "synthetic.json").read_text())
print("LIVE: paid Bedrock calls" if LIVE else "OFFLINE: synthetic teaching fixtures; no AWS calls")


## Worked example: evidence before scores
A result stores scenario/model identity, response, assertions, elapsed seconds, token usage, errors, completed tool calls, source, optional response cost, and separate grading usage. Offline fixtures have no measured token usage or latency. Missing cost remains `None`.

The following request is synthetic by default. Live mode makes **one response request**. Graders and embeddings in later labs add paid requests; thresholds are checks after completion, not spending caps.


In [ ]:
result = generate(api, "setup-01", "Reply with the word ready.", "ready")
if not result.errors:
    result.assertions.append(check("contains-ready", contains(result.response, "ready")))
from dataclasses import asdict
asdict(result)


## Inspect the mechanism
`Bedrock.text()` sends `messages` to Converse, extracts text blocks, and records response usage and wall time. Offline mode returns the supplied fixture without constructing a Boto3 client. Errors are recorded instead of being counted as correct answers.


In [ ]:
import inspect
print(inspect.getsource(Bedrock.text))
export_results([result], ROOT / "results/01_setup")
print("Exported JSON and CSV under results/")


## Exercise
Create a second `Result` with an answer that does not contain `ready`; apply the same check and export both results. Add a different assertion that would detect an empty response.

**Hint:** reuse `check()` and `contains()`; a score is a value from 0 to 1.

**Checkpoints:** Which fields describe observation, and which express an evaluator's decision? Why is an unavailable cost different from zero? What does a substring pass fail to establish?

Next: [Translation evaluation](02_translation_evaluation.ipynb).
